In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("giang.csv")

In [4]:
# Clean and preprocess df (giang.csv)
# Assumes pandas already imported and df already loaded

# 1. Drop unnamed index column if present
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

# 2. Strip whitespace from object (string) columns
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    df[col] = df[col].str.strip()

# 3. Convert numeric-like columns to numeric (coerce errors to NaN)
num_cols = ['zpid', 'price', 'latitude', 'longitude', 'bathrooms', 'bedrooms', 'living_area', 'lot_area', 'days_on_zillow']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Normalize lot area to square feet (new column: lot_area_sqft)
if 'lot_area' in df.columns:
    df['lot_area_sqft'] = df['lot_area']  # start with existing values
    if 'lot_area_unit' in df.columns:
        mask_acres = df['lot_area_unit'].fillna('').str.lower().str.contains('acre')
        df.loc[mask_acres, 'lot_area_sqft'] = df.loc[mask_acres, 'lot_area'] * 43560
    df['lot_area_sqft'] = pd.to_numeric(df['lot_area_sqft'], errors='coerce')

# 5. Fill missing categorical values with 'Unknown'
for col in ['status', 'type', 'city', 'state', 'lot_area_unit']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# 6. Remove exact duplicate listings by zpid (if present)
if 'zpid' in df.columns:
    df.drop_duplicates(subset=['zpid'], inplace=True)

# 7. Drop rows with no key numeric info (all of price, latitude, longitude missing)
drop_subset = [c for c in ['price', 'latitude', 'longitude'] if c in df.columns]
if drop_subset:
    df.dropna(subset=drop_subset, how='all', inplace=True)

# 8. Reset index
df.reset_index(drop=True, inplace=True)

# 9. Quick summary
print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 5756 entries, 0 to 5755
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   zpid            5756 non-null   float64
 1   price           5756 non-null   float64
 2   status          5756 non-null   str    
 3   type            5756 non-null   str    
 4   city            5756 non-null   str    
 5   state           5756 non-null   str    
 6   latitude        5756 non-null   float64
 7   longitude       5756 non-null   float64
 8   bathrooms       4792 non-null   float64
 9   bedrooms        4707 non-null   float64
 10  living_area     4715 non-null   float64
 11  lot_area        5200 non-null   float64
 12  lot_area_unit   5756 non-null   str    
 13  days_on_zillow  5756 non-null   float64
 14  lot_area_sqft   5200 non-null   float64
dtypes: float64(10), str(5)
memory usage: 674.7 KB
None


C:\Users\giangpham\AppData\Local\Temp\ipykernel_3388\1597986659.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include='object').columns


,zpid,price,status,type,city,state,latitude,longitude,bathrooms,bedrooms,living_area,lot_area,lot_area_unit,days_on_zillow,lot_area_sqft
0,2.073568e+09,4980000.0,House for sale,SINGLE_FAMILY,West Hollywood,CA,34.100803,-118.380570,5.0,4.0,4126.0,4922.0000,sqft,7.908150e+08,4922.000
1,1.987515e+07,1215000.0,House for sale,SINGLE_FAMILY,Woodland Hills,CA,34.180298,-118.642500,2.0,3.0,1825.0,7840.8000,sqft,1.020029e+09,7840.800
2,2.053261e+07,2629000.0,House for sale,SINGLE_FAMILY,Beverly Hills,CA,34.112854,-118.433876,4.0,4.0,3019.0,0.9959,acres,4.203270e+08,43381.404
3,2.093480e+07,400000.0,Multi-family home for sale,MULTI_FAMILY,Los Angeles,CA,33.978700,-118.295680,2.0,3.0,944.0,4268.8800,sqft,9.995380e+08,4268.880
4,2.131354e+07,849000.0,House for sale,SINGLE_FAMILY,San Pedro,CA,33.713604,-118.291880,2.0,2.0,1154.0,5002.0000,sqft,6.436090e+08,5002.000


In [ ]:
# Fill missing numeric values with median
numeric_cols = ['price', 'latitude', 'longitude', 'bathrooms', 'bedrooms', 'living_area', 'lot_area', 'days_on_zillow', 'lot_area_sqft']
for col in numeric_cols:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

# Check lot_area_sqft values for reasonableness
print("lot_area_sqft statistics:")
print(df['lot_area_sqft'].describe())
print("\nAny lot_area_sqft values <= 0?", (df['lot_area_sqft'] <= 0).any())
print("Any lot_area_sqft values > 1,000,000?", (df['lot_area_sqft'] > 1000000).any())

# Show sample of lot_area_sqft values
print("\nSample lot_area_sqft values:")
print(df[['lot_area', 'lot_area_unit', 'lot_area_sqft']].head(10))